In [ ]:
# vanilla data science stuff
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# single-cell and spatial omics packages
import anndata as ad 
import scanpy as sc
import squidpy as sq

### The data

- Healthy human tonsil
- Imaged using Cyclic Immunofluorescence 
- Generated for the Human Tumor Atlas Network (HTAN)

Find the original dataset here: https://humantumoratlas.org/ 

The specific feature table (`cycif_tonsil.csv`) used in this notebook can be downloaded from this public Galaxy history: 

https://cancer.usegalaxy.org/u/watsocam/h/bgmp-spatial-omics-demo-tonsil-cycif 

The Galaxy history also has Vitessce dashboards that can be used to view the image, segmentation mask, and downstream quantified data

### Read feature table and convert to anndata format

In [ ]:
# read as dataframe (you probably need to change the path and filename)
df = pd.read_csv('cycif_tonsil.csv', index_col='CellID')

In [ ]:
# inspect dataframe
df

In [ ]:
# determine where to split cols into X and obs
print(df.columns)

In [ ]:
# create anndata object with markers in X matrix and observational columns in obs
adata = ad.AnnData(X=df.iloc[:, 0:25], obs=df.iloc[:, 25:])

In [ ]:
adata

In [ ]:
# important: be careful of indices with anndata
# best practice is to subset the entire anndata object using the obs_names as common indices for X and obs
print(adata.obs_names)
print(adata.X.index)

In [ ]:
# marker names are in var_names
print(adata.var_names)

In [ ]:
# remove DAPI and H3 from feature table since they're unhelpful for cell typing
# remove Lamin, pH3, and H2ax due to tissue loss in final round
# remove FOXP3, and PDL1 due to bubble
features_to_exclude = ['Hoechst1', 'LaminAC_488', 'pH3_555', 'H2ax_647', 'H3_PE', 'FOXP3_570', 'PDL1_647']
adata_filtered = adata[:, ~adata.var_names.isin(features_to_exclude)].copy()
print(adata_filtered)

### Plot cell spatial coordinates

In [ ]:
fig,ax = plt.subplots()
sns.scatterplot(
    x=adata_filtered.obs['X_centroid'],
    y=adata_filtered.obs['Y_centroid'],
    ax=ax)

plt.show()

In [ ]:
# make the centroids a reasonable size and set the aspect ratio
fig,ax = plt.subplots()
sns.scatterplot(
    x=adata_filtered.obs['X_centroid'],
    y=adata_filtered.obs['Y_centroid'],
    s=1,
    ax=ax)

ax.set_aspect('equal')

plt.show()

In [ ]:
# overlay the intensity of a marker
fig,ax = plt.subplots()
sns.scatterplot(
    x=adata_filtered.obs['X_centroid'],
    y=adata_filtered.obs['Y_centroid'],
    c=adata_filtered.X[:, adata_filtered.var_names.isin(['CD20_488'])],
    s=1,
    ax=ax)

ax.set_aspect('equal')

plt.show()

In [ ]:
# set the origin to the upper left corner
fig,ax = plt.subplots()
sns.scatterplot(
    x=adata_filtered.obs['X_centroid'],
    y=adata_filtered.obs['Y_centroid'],
    c=adata_filtered.X[:, adata_filtered.var_names.isin(['CD20_488'])],
    s=1,
    ax=ax)

ax.invert_yaxis()
ax.set_aspect('equal')

plt.show()

### Explore cell molecular states

In [ ]:
# quick look at marker histograms
sns.kdeplot(adata_filtered.X[:, adata_filtered.var_names.isin(['CD20_488'])])

In [ ]:
# marker intensities tend to be highly skewed and have a lot of outliers
# log1p transform pulls distribution more towards normal, which will help a lot with UMAP and clustering algorithms
sns.kdeplot(np.log1p(adata_filtered.X[:, adata_filtered.var_names.isin(['CD20_488'])]))

In [ ]:
# apply log1p to all features
# NOTE: scanpy and other sc-verse tools change the state of a variable in-place
# be careful with some of scanpy's normalization methods, protein intensities are not counts
sc.pp.log1p(adata_filtered)

In [ ]:
# compute FEATURE SPACE neighbors
sc.pp.neighbors(adata_filtered)

In [ ]:
# run UMAP algorithm
# in scRNA, you'd probably run PCA first to bring the features into the 10-50 or so range but we only have 18 markers here to start with
sc.tl.umap(adata_filtered)

In [ ]:
# visualize marker intensities on UMAP
sc.pl.umap(adata_filtered, color=adata_filtered.var_names)

In [ ]:
# leiden clustering with default resolution
sc.tl.leiden(adata_filtered, resolution=1, flavor='igraph', n_iterations=2)

In [ ]:
sc.pl.umap(adata_filtered, color='leiden')

In [ ]:
# visualize the mean marker intensities and total cell counts by leiden cluster
mp = sc.pl.matrixplot(
    adata_filtered,
    var_names=adata_filtered.var_names,
    groupby='leiden',
    dendrogram=True,
    standard_scale='var',
    return_fig=True)

mp.add_totals().style(edge_color='black').show()

In [ ]:
# Ecad and catenin higher in epithelial cells lining the crypt and exposed to the lumen
# Epithelial cells closer to B-cells have higher CD20 due to fluorescent spillover
adata_filtered.obsm['spatial'] = np.array(adata_filtered.obs[['X_centroid', 'Y_centroid']])
sq.pl.spatial_scatter(adata_filtered, shape=None, color='leiden', groups=['4','5'])

### Assign clusters to cell types

In [ ]:
# probably not the best annotations ever
leiden_to_cell_type = {
    '4' : 'Epithelial',
    '5' : 'Epithelial (Ecad/Catenin High)',
    '11': 'Dendritic cells',
    '12': 'Memory CD4 T cells',
    '16': 'Memory CD4 T cells',
    '14': 'Naive CD4 T cells',
    '13': 'T cells (other)',
    '15': 'Tissue resident memory CD8 T cells',
    '2' : 'Endothelial',
    '6' : 'Germinal center B cells',
    '18': 'Germinal center B cells',
    '10': 'Intraepithelial B cells',
    '17': 'Germinal center B cells',
    '9' : 'Macrophages',
    '0' : 'B cells',
    '7' : 'B cells',
    '8' : 'Stroma',
    '1' : 'B cells',
    '3' : 'B cells',
}

# map cell types
adata_filtered.obs['cell_type'] = adata_filtered.obs['leiden'].map(leiden_to_cell_type)

In [ ]:
# visualize the mean marker intensities and total cell counts by cell type
mp = sc.pl.matrixplot(
    adata_filtered,
    var_names=adata_filtered.var_names,
    groupby='cell_type',
    dendrogram=True,
    standard_scale='var',
    return_fig=True)

mp.add_totals().style(edge_color='black').show()

In [ ]:
# ignore clearing uns here. accomodating a current quirk of our Galaxy Vitessce implementation
adata_to_write = adata_filtered.copy()
adata_to_write.uns = {}
adata_to_write.write_h5ad('tonsil_cell_types.h5ad')

### Spatial analysis (finally)

In [ ]:
# squidpy provides an easier way to create spatial plots
sq.pl.spatial_scatter(adata_filtered, shape=None, color='cell_type')

In [ ]:
# compute a spatial neighborhood graph - not to be confused with the feature-space neighbors computed earlier using scanpy
cycif_physical_resolution = 0.65 # units is microns/pixel
radius_max = 10 / 0.65 # converts our desired distance in microns to pixel space using resolution
sq.gr.spatial_neighbors(adata_filtered, coord_type='generic', radius=(0, radius_max), delaunay=True)

In [ ]:
# check out the graph on a small subset of the image
sq.pl.spatial_scatter(
    adata_filtered, 
    shape=None, 
    color='cell_type',
    size=1, 
    edges_width=.1, 
    crop_coord=(2000,1000,2500,1500),
    # edges_color='black', 
    connectivity_key='spatial_connectivities',
    scalebar_dx=0.65,
    scalebar_units='um',
    figsize=(6,6),
    dpi=300)

In [ ]:
# take a look at which cell types are frequently neighbors in the tissue, row normalized
sq.gr.interaction_matrix(adata_filtered, cluster_key='cell_type', normalized=True)
sq.pl.interaction_matrix(adata_filtered, cluster_key='cell_type')

In [ ]:
# compute spatial autocorrelation of all markers
sq.gr.spatial_autocorr(
    adata_filtered,
    mode="moran",
    genes=adata_filtered.var_names,
    n_perms=100,
    n_jobs=1,
)

In [ ]:
# it's really important to think about the null hypothesis for Moran's I and whether that is appropriate in this context
adata_filtered.uns['moranI']

In [ ]:
# look at two markers with high and low spatial autocorrelation
sq.pl.spatial_scatter(
    adata_filtered, 
    shape=None, 
    color=['Keratin_570', 'CD8a_488'],
    size=1, 
    scalebar_dx=0.65,
    scalebar_units='um',
    figsize=(6,6),
    dpi=300)